<a href="https://www.kaggle.com/code/piyushxo19/training-a-neural-network-with-dataloader-class?scriptVersionId=318210676" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [324]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/ssssws/global-inflation-dynamics-post-covid-20202024/global_inflation_post_covid.csv


In [325]:
import numpy as np 
import pandas as pd 
import torch 
import torch.nn as nn 
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder


In [326]:
df=pd.read_csv("/kaggle/input/datasets/ssssws/global-inflation-dynamics-post-covid-20202024/global_inflation_post_covid.csv")


In [327]:
print("DATASET BEFORE ",df.head())


df=df.drop(columns="date")
print("DATASET AFTER DROPPING THE DATE COLUMNS",df.head())


DATASET BEFORE    country     date  inflation_rate  interest_rate  oil_price  gdp_growth  \
0     USA  2020-01            0.33           5.76      34.44        0.34   
1     USA  2020-02            4.12           4.33      44.75        0.02   
2     USA  2020-03            2.75           4.24      52.80        0.30   
3     USA  2020-04            4.36           4.52      39.19        2.28   
4     USA  2020-05            2.73           3.17      41.09        3.05   

   unemployment_rate  money_supply_m2  exchange_rate_usd  food_price_index  \
0               5.97          10558.0              69.95             94.29   
1               5.80           6952.0              71.03             78.60   
2               7.45          10244.0              71.00            102.91   
3               5.23           9989.0              83.52            110.92   
4               5.96           7982.0              70.10             86.53   

   supply_chain_index  
0                3.66  
1         

In [328]:
le=LabelEncoder()
df['country']=le.fit_transform(df['country'])
print(df.head())
print(df.country.nunique())


   country  inflation_rate  interest_rate  oil_price  gdp_growth  \
0       17            0.33           5.76      34.44        0.34   
1       17            4.12           4.33      44.75        0.02   
2       17            2.75           4.24      52.80        0.30   
3       17            4.36           4.52      39.19        2.28   
4       17            2.73           3.17      41.09        3.05   

   unemployment_rate  money_supply_m2  exchange_rate_usd  food_price_index  \
0               5.97          10558.0              69.95             94.29   
1               5.80           6952.0              71.03             78.60   
2               7.45          10244.0              71.00            102.91   
3               5.23           9989.0              83.52            110.92   
4               5.96           7982.0              70.10             86.53   

   supply_chain_index  
0                3.66  
1                3.91  
2                3.70  
3                3.23  
4 

In [329]:
print(df.columns)

Index(['country', 'inflation_rate', 'interest_rate', 'oil_price', 'gdp_growth',
       'unemployment_rate', 'money_supply_m2', 'exchange_rate_usd',
       'food_price_index', 'supply_chain_index'],
      dtype='object')


In [330]:
feature_cols=['country', 'interest_rate', 'oil_price', 'gdp_growth',
       'unemployment_rate', 'money_supply_m2', 'exchange_rate_usd',
       'food_price_index', 'supply_chain_index']

In [331]:
target_cols=["inflation_rate"]


In [332]:
train_val_df,test_df=train_test_split(df,test_size=0.3,random_state=42)
train_df,val_df=train_test_split(train_val_df,test_size=0.15,random_state=42)

#reseting the index value 
train_df=train_df.reset_index(drop=True)
val_df=val_df.reset_index(drop=True)
test_df=test_df.reset_index(drop=True)


In [333]:
#Standardization of the columns
from sklearn.preprocessing import StandardScaler
scaler=StandardScaler()

train_df[feature_cols]=scaler.fit_transform(train_df[feature_cols])
val_df[feature_cols]=scaler.transform(val_df[feature_cols])
test_df[feature_cols]=scaler.transform(test_df[feature_cols])

In [334]:
from torch.utils.data import Dataset

In [335]:
class Global_inflation(Dataset):
    def __init__(self,df,feature_cols,target_cols):
        self.X=df[feature_cols].values.astype("float32")
        self.y=df[target_cols].values.astype("float32")
    
    def __len__(self):
        return len(self.X)

    def __getitem__(self,idx):
        features= torch.tensor(self.X[idx])
        labels=torch.tensor(self.y[idx])
        return features ,labels

In [336]:
# creating dataset class objects 
train_dataset= Global_inflation(train_df,feature_cols,target_cols)
val_dataset=Global_inflation(val_df,feature_cols,target_cols)
test_dataset=Global_inflation(test_df,feature_cols,target_cols)


In [337]:
# creating the dataloader class
from torch.utils.data import DataLoader
train_loader=DataLoader(train_dataset,batch_size=32,shuffle=True)
val_loader=DataLoader(val_dataset,batch_size=32,shuffle=True)
test_loader=DataLoader(test_dataset,batch_size=32,shuffle=True)

In [338]:
import torch.nn as nn 
class Global_inflation_model(nn.Module):
    def __init__(self,no_of_feature=9,h1=64,h2=32):
        super().__init__()
        self.network=nn.Sequential(
            nn.Linear(no_of_feature,h1),
            nn.ReLU(),
            nn.Linear(h1,h2),
            nn.ReLU(),
            nn.Linear(h2,1)
            
        )
    def forward(self,X):
        return self.network(X)

In [339]:
model=Global_inflation_model()
criterion=nn.MSELoss()
optimizer= torch.optim.Adam(model.parameters(),lr=0.01)


In [340]:
epochs=100
for epoch in range(epochs):
    model.train() #telling model that it is in training phase
    train_loss=0
    for X_batch,y_batch in train_loader:
        pred=model(X_batch)
        loss=criterion(pred,y_batch)
        #backward pass
        optimizer.zero_grad()
        loss.backward()
        #updating the weights 
        optimizer.step()
        train_loss+=loss.item()

    print(f"Epoch {epoch+1}/{epochs} | Train Loss: {train_loss/len(train_loader):.4f}")

Epoch 1/100 | Train Loss: 1.5926
Epoch 2/100 | Train Loss: 1.0480
Epoch 3/100 | Train Loss: 0.8321
Epoch 4/100 | Train Loss: 0.6821
Epoch 5/100 | Train Loss: 0.5804
Epoch 6/100 | Train Loss: 0.5277
Epoch 7/100 | Train Loss: 0.4957
Epoch 8/100 | Train Loss: 0.4637
Epoch 9/100 | Train Loss: 0.4523
Epoch 10/100 | Train Loss: 0.4412
Epoch 11/100 | Train Loss: 0.4292
Epoch 12/100 | Train Loss: 0.4227
Epoch 13/100 | Train Loss: 0.4137
Epoch 14/100 | Train Loss: 0.4110
Epoch 15/100 | Train Loss: 0.4074
Epoch 16/100 | Train Loss: 0.4069
Epoch 17/100 | Train Loss: 0.4014
Epoch 18/100 | Train Loss: 0.3998
Epoch 19/100 | Train Loss: 0.4020
Epoch 20/100 | Train Loss: 0.3934
Epoch 21/100 | Train Loss: 0.3922
Epoch 22/100 | Train Loss: 0.3918
Epoch 23/100 | Train Loss: 0.3883
Epoch 24/100 | Train Loss: 0.3864
Epoch 25/100 | Train Loss: 0.3856
Epoch 26/100 | Train Loss: 0.3847
Epoch 27/100 | Train Loss: 0.3869
Epoch 28/100 | Train Loss: 0.3806
Epoch 29/100 | Train Loss: 0.3841
Epoch 30/100 | Train Lo

In [345]:
model.eval()                                  # tell model its in eval mode
val_loss = 0
with torch.no_grad():                         # no gradients needed here
    for X_batch, y_batch in val_loader:
        preds     = model(X_batch)
        loss      = criterion(preds, y_batch)
        val_loss += loss.item()
# ── Print progress ──────────────────────────
print(f"Epoch {epoch+1:02d}/{epochs} | "
      f"Train Loss: {train_loss/len(train_loader):.4f} | "
      f"Val Loss: {val_loss/len(val_loader):.4f}")
# ── TEST PHASE (only once, after training is done) ───────────────
model.eval()
test_loss = 0
with torch.no_grad():
    for X_batch, y_batch in test_loader:
        preds      = model(X_batch)
        loss       = criterion(preds, y_batch)
        test_loss += loss.item()
print(f"\nFinal Test Loss: {test_loss/len(test_loader):.4f}")

Epoch 100/100 | Train Loss: 0.3618 | Val Loss: 0.3455

Final Test Loss: 0.3533


In [347]:
from sklearn.metrics import mean_absolute_error, r2_score
import numpy as np

model.eval()
all_preds  = []
all_labels = []

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        preds  = model(X_batch).numpy()
        labels = y_batch.numpy()

        all_preds.extend(preds)
        all_labels.extend(labels)

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)

mae = mean_absolute_error(all_labels, all_preds)
r2  = r2_score(all_labels, all_preds)

print(f"MAE : {mae:.4f}")
print(f"R2  : {r2:.4f}")

MAE : 0.4710
R2  : 0.8347
